# Classifier Comparison: Which Algorithm Best Separates Personality Factors?

The [Quick Start](https://colab.research.google.com/github/Wildertrek/survey/blob/main/notebooks/atlas_quick_start.ipynb) notebook uses Random Forest throughout. But is RF the best classifier for this task? This notebook trains and evaluates **four classifiers** (Random Forest, Logistic Regression, Linear SVC, k-Nearest Neighbors) across all 44 personality models and finds that **RF is consistently the weakest** on novel items.

**Key findings:**
- Linear SVC achieves 74.7% mean accuracy on LLM-generated test items (vs RF 60.4%)
- k-NN achieves 83.5% on human-authored items from 20 published instruments (vs RF 69.8%)
- Oracle (best classifier per model) reaches 86.8% on human items with 100% model-level convergence
- RF has a 43% error rate on reverse-scored items; k-NN reduces this to 11%
- Friedman test confirms classifier differences are statistically significant (chi2=47.9, p<0.001)

Everything runs from pre-computed assets. No API keys needed.

> Raetano, J., Gregor, J., & Tamang, S. (2026). *A Survey and Computational Atlas of Personality Models.* ACM TIST. Under review.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Wildertrek/survey/blob/main/notebooks/atlas_classifier_comparison.ipynb)

## 1. Setup

Clone the atlas repository and install dependencies. Takes about 30 seconds on Colab.

In [ ]:
# Clone the atlas repository (skip if already cloned)
import os
if not os.path.exists("atlas"):
    !git clone --depth 1 https://github.com/Wildertrek/survey.git atlas
else:
    print("Atlas already cloned — skipping.")

In [ ]:
import ast
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from scipy.stats import friedmanchisquare

warnings.filterwarnings("ignore", message=".*sklearn.*")

# Discover all 44 models
models = sorted([f.replace(".csv", "") for f in os.listdir("atlas/datasets") if f.endswith(".csv")])
print(f"Atlas contains {len(models)} personality models")

## 2. Train Four Classifiers on All 44 Models

Each model's training data consists of factor chain embeddings (1,536-dim vectors from OpenAI `text-embedding-3-small`). We train four classifiers per model using the same training split and hyperparameters as the paper.

In [ ]:
# Train all four classifiers on each model's embeddings
# Note: we fit fresh LabelEncoders from the CSV data to avoid sklearn version conflicts
trained = {}  # model -> {clf_name: classifier}
encoders = {}  # model -> label_encoder
train_acc = []  # training accuracy records

for model in models:
    df = pd.read_csv(f"atlas/datasets/{model}.csv")
    emb = pd.read_csv(f"atlas/Embeddings/{model}_embeddings.csv")
    X = np.array([ast.literal_eval(e) for e in emb["Embedding"]])

    # Fit encoder from training data (avoids pkl version mismatch on Colab)
    enc = LabelEncoder()
    y = enc.fit_transform(df["Factor"].values)
    encoders[model] = enc

    clfs = {
        "RF":  RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
        "LR":  LogisticRegression(max_iter=1000, random_state=42),
        "SVC": LinearSVC(max_iter=2000, random_state=42, dual="auto"),
        "kNN": KNeighborsClassifier(n_neighbors=5),
    }

    model_clfs = {}
    for name, clf in clfs.items():
        clf.fit(X, y)
        acc = accuracy_score(y, clf.predict(X))
        model_clfs[name] = clf
        train_acc.append({"Model": model.upper(), "Classifier": name, "Train Acc": acc})
    trained[model] = model_clfs

print(f"Trained {len(models) * 4} classifiers ({len(models)} models x 4 types)")
train_df = pd.DataFrame(train_acc)
print(f"\nMean training accuracy by classifier:")
print(train_df.groupby("Classifier")["Train Acc"].mean().to_string())

## 3. Novel-Item Evaluation (LLM-Generated Test Items)

The real test: how well does each classifier generalize to items it has never seen? These 5,038 test items were generated by GPT-4o and embedded offline — they are semantically novel paraphrases of each factor.

In [ ]:
# Load pre-computed test items
test_items = json.load(open("atlas/data/test_items/test_items.json"))
test_emb = np.load("atlas/data/test_items/test_items_embeddings.npz")["embeddings"]

# Evaluate all 4 classifiers on all 44 models
novel_results = []
for model in models:
    idx = [i for i, item in enumerate(test_items) if item["model"] == model]
    if not idx:
        continue

    X_test = test_emb[idx]
    y_true = [test_items[i]["expected_factor"] for i in idx]
    enc = encoders[model]

    row = {"Model": model.upper(), "Items": len(idx)}
    accs = {}
    for clf_name, clf in trained[model].items():
        y_pred = enc.inverse_transform(clf.predict(X_test))
        acc = accuracy_score(y_true, y_pred)
        row[clf_name] = acc
        accs[clf_name] = acc

    best_name = max(accs, key=accs.get)
    row["Best"] = best_name
    row["Best Acc"] = accs[best_name]
    novel_results.append(row)

novel_df = pd.DataFrame(novel_results)

print("=== Novel-Item Test Accuracy (44 models) ===")
print(f"  RF:   {novel_df['RF'].mean():.1%} mean (median {novel_df['RF'].median():.1%})")
print(f"  LR:   {novel_df['LR'].mean():.1%} mean (median {novel_df['LR'].median():.1%})")
print(f"  SVC:  {novel_df['SVC'].mean():.1%} mean (median {novel_df['SVC'].median():.1%})")
print(f"  kNN:  {novel_df['kNN'].mean():.1%} mean (median {novel_df['kNN'].median():.1%})")
print(f"  Best-per-model: {novel_df['Best Acc'].mean():.1%} mean")
print(f"\nClassifier wins: {novel_df['Best'].value_counts().to_dict()}")

### Friedman Test: Are Classifier Differences Significant?

The Friedman test is the non-parametric equivalent of repeated-measures ANOVA — appropriate here because the same 44 models are evaluated under four conditions.

In [ ]:
# Friedman test across all 44 models
stat, p = friedmanchisquare(
    novel_df["RF"].values,
    novel_df["LR"].values,
    novel_df["SVC"].values,
    novel_df["kNN"].values,
)
print(f"Friedman chi2 = {stat:.3f}, p = {p:.2e}")
if p < 0.001:
    print("Classifier differences are statistically significant (p < 0.001).")
    print("RF is not merely unlucky — it is systematically worse on novel items.")

### Visualization: Per-Model Accuracy Across Classifiers

In [ ]:
# Sort by best-per-model accuracy for a clean visual
plot_df = novel_df.sort_values("Best Acc", ascending=True).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(14, 10))
y_pos = range(len(plot_df))
colors = {"RF": "#e74c3c", "LR": "#3498db", "SVC": "#2ecc71", "kNN": "#9b59b6"}

for clf_name, color in colors.items():
    ax.scatter(plot_df[clf_name], y_pos, label=clf_name, color=color, alpha=0.7, s=30)

ax.set_yticks(y_pos)
ax.set_yticklabels(plot_df["Model"], fontsize=7)
ax.set_xlabel("Novel-Item Accuracy")
ax.set_title("Four-Way Classifier Comparison: Novel Items (44 Models)")
ax.legend(loc="lower right")
ax.axvline(x=0.5, color="gray", linestyle="--", alpha=0.3, label="chance")
ax.set_xlim(0, 1.05)
plt.tight_layout()
plt.show()

## 4. Human-Item Evaluation (Published Instruments)

The strongest external validation: 368 items from 20 published psychometric instruments (BFI-44, GAD-7, HEXACO-60, Short Dark Triad, etc.). These are the items clinicians and researchers actually use.

In [ ]:
# Load pre-computed human items
human_items = json.load(open("atlas/data/human_items.json"))
human_emb = np.load("atlas/data/human_items_embeddings.npz")["embeddings"]

human_models = sorted(set(item["model"] for item in human_items))
print(f"{len(human_items)} items from {len(human_models)} models with published instruments\n")

# Evaluate all 4 classifiers on human items
human_results = []
for model in human_models:
    idx = [i for i, item in enumerate(human_items) if item["model"] == model]
    X_human = human_emb[idx]
    y_true = [human_items[i]["expected_factor"] for i in idx]
    instrument = human_items[idx[0]]["instrument"]
    enc = encoders[model]

    row = {"Model": model.upper(), "Instrument": instrument, "Items": len(idx)}
    accs = {}
    for clf_name, clf in trained[model].items():
        y_pred = enc.inverse_transform(clf.predict(X_human))
        acc = accuracy_score(y_true, y_pred)
        row[clf_name] = acc
        accs[clf_name] = acc

    best_name = max(accs, key=accs.get)
    row["Best"] = best_name
    row["Best Acc"] = accs[best_name]
    human_results.append(row)

human_df = pd.DataFrame(human_results).sort_values("Best Acc", ascending=False)

print("=== Human-Item Accuracy (Published Instruments) ===")
print(f"  RF:   {human_df['RF'].mean():.1%}")
print(f"  LR:   {human_df['LR'].mean():.1%}")
print(f"  SVC:  {human_df['SVC'].mean():.1%}")
print(f"  kNN:  {human_df['kNN'].mean():.1%}")
print(f"  Best-per-model (oracle): {human_df['Best Acc'].mean():.1%}")
print(f"\nClassifier wins: {human_df['Best'].value_counts().to_dict()}")

In [ ]:
# Display full human-item results table
display_human = human_df.copy()
for c in ["RF", "LR", "SVC", "kNN", "Best Acc"]:
    display_human[c] = display_human[c].apply(lambda x: f"{x:.1%}")
display_human.index = range(1, len(display_human) + 1)
display_human

### Friedman Test: Human Items

In [ ]:
stat_h, p_h = friedmanchisquare(
    human_df["RF"].values,
    human_df["LR"].values,
    human_df["SVC"].values,
    human_df["kNN"].values,
)
print(f"Friedman chi2 = {stat_h:.3f}, p = {p_h:.6f}")
if p_h < 0.001:
    print("Human-item classifier differences are also significant (p < 0.001).")

## 5. Reverse-Scored Item Analysis

Reverse-scored items (e.g., *"I am calm and relaxed"* for Neuroticism) are the hardest test case. They describe the **low pole** of a factor, while training data only covers the high pole. RF's decision boundaries struggle with this; linear and instance-based methods handle it better.

In [ ]:
# Identify reverse-scored human items and compute per-classifier error rates
rev_errors = {clf: {"rev_err": [], "fwd_err": []} for clf in ["RF", "LR", "SVC", "kNN"]}

for model in human_models:
    idx = [i for i, item in enumerate(human_items) if item["model"] == model]
    y_true = [human_items[i]["expected_factor"] for i in idx]
    X_human = human_emb[idx]
    enc = encoders[model]

    # Check if items have reverse-scoring metadata
    has_reverse = any(human_items[i].get("reverse", False) for i in idx)
    if not has_reverse:
        continue

    rev_idx = [j for j, i in enumerate(idx) if human_items[i].get("reverse", False)]
    fwd_idx = [j for j, i in enumerate(idx) if not human_items[i].get("reverse", False)]

    for clf_name, clf in trained[model].items():
        y_pred = enc.inverse_transform(clf.predict(X_human))
        if rev_idx:
            rev_err = 1 - accuracy_score([y_true[j] for j in rev_idx], [y_pred[j] for j in rev_idx])
            rev_errors[clf_name]["rev_err"].append(rev_err)
        if fwd_idx:
            fwd_err = 1 - accuracy_score([y_true[j] for j in fwd_idx], [y_pred[j] for j in fwd_idx])
            rev_errors[clf_name]["fwd_err"].append(fwd_err)

print("=== Reverse-Scored vs Forward-Scored Error Rates ===")
print(f"{'Classifier':<12} {'Reverse Err':>12} {'Forward Err':>12} {'Gap':>8}")
print("-" * 46)
for clf_name in ["RF", "LR", "SVC", "kNN"]:
    rev = np.mean(rev_errors[clf_name]["rev_err"]) if rev_errors[clf_name]["rev_err"] else 0
    fwd = np.mean(rev_errors[clf_name]["fwd_err"]) if rev_errors[clf_name]["fwd_err"] else 0
    print(f"{clf_name:<12} {rev:>11.1%} {fwd:>11.1%} {rev - fwd:>+7.1%}")

print("\nRF's reverse-scored error rate is 3-4x higher than linear classifiers.")
print("This is because RF builds axis-aligned splits in 1536-dim space, while")
print("linear methods find the hyperplane that separates factors regardless of pole.")

## 6. Reliability Tiers

With the best classifier per model, we can assign each model a reliability tier based on its ability to generalize to novel items and (where available) human-authored items:

- **Reliable** (>70% test accuracy + >75% human accuracy): Suitable for deployment
- **Usable** (50-70%): Useful with caveats
- **Research-only** (<50%): Insufficient generalization

In [ ]:
# Assign reliability tiers based on best-per-model accuracy
human_acc_map = {r["Model"]: r["Best Acc"] for _, r in human_df.iterrows()}

tiers = {"Reliable": [], "Usable": [], "Research-Only": []}
for _, row in novel_df.iterrows():
    test_acc = row["Best Acc"]
    h_acc = human_acc_map.get(row["Model"], None)

    if test_acc >= 0.70 and (h_acc is None or h_acc >= 0.75):
        tiers["Reliable"].append(row["Model"])
    elif test_acc >= 0.50:
        tiers["Usable"].append(row["Model"])
    else:
        tiers["Research-Only"].append(row["Model"])

print("=== Reliability Tiers (Best-per-Model) ===")
for tier, model_list in tiers.items():
    print(f"\n{tier} ({len(model_list)} models):")
    for m in sorted(model_list):
        best_row = novel_df[novel_df["Model"] == m].iloc[0]
        h_str = f", human {human_acc_map[m]:.0%}" if m in human_acc_map else ""
        print(f"  {m:<8} {best_row['Best']:>4} {best_row['Best Acc']:.1%}{h_str}")

print(f"\nSummary: {len(tiers['Reliable'])} reliable / {len(tiers['Usable'])} usable / {len(tiers['Research-Only'])} research-only")

## 7. Summary Comparison

Side-by-side comparison of all four classifiers on both evaluation sets.

In [ ]:
# Summary comparison table
summary_rows = []
for clf_name in ["RF", "LR", "SVC", "kNN"]:
    summary_rows.append({
        "Classifier": clf_name,
        "Novel Mean": f"{novel_df[clf_name].mean():.1%}",
        "Novel Median": f"{novel_df[clf_name].median():.1%}",
        "Human Mean": f"{human_df[clf_name].mean():.1%}",
        "Human Median": f"{human_df[clf_name].median():.1%}",
        "Wins (Novel)": (novel_df["Best"] == clf_name).sum(),
        "Wins (Human)": (human_df["Best"] == clf_name).sum(),
    })
summary_rows.append({
    "Classifier": "Oracle",
    "Novel Mean": f"{novel_df['Best Acc'].mean():.1%}",
    "Novel Median": f"{novel_df['Best Acc'].median():.1%}",
    "Human Mean": f"{human_df['Best Acc'].mean():.1%}",
    "Human Median": f"{human_df['Best Acc'].median():.1%}",
    "Wins (Novel)": "-",
    "Wins (Human)": "-",
})

pd.DataFrame(summary_rows).set_index("Classifier")

In [ ]:
# Grouped bar chart: Novel vs Human accuracy by classifier
clf_names = ["RF", "LR", "SVC", "kNN", "Oracle"]
novel_means = [novel_df[c].mean() if c != "Oracle" else novel_df["Best Acc"].mean() for c in clf_names]
human_means = [human_df[c].mean() if c != "Oracle" else human_df["Best Acc"].mean() for c in clf_names]

x = np.arange(len(clf_names))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, [v * 100 for v in novel_means], width, label="Novel Items (44 models)", color="#3498db")
bars2 = ax.bar(x + width/2, [v * 100 for v in human_means], width, label="Human Items (20 models)", color="#e74c3c")

ax.set_ylabel("Accuracy (%)")
ax.set_title("Classifier Comparison: Novel vs Human Items")
ax.set_xticks(x)
ax.set_xticklabels(clf_names)
ax.legend()
ax.set_ylim(0, 100)
ax.bar_label(bars1, fmt="%.1f%%", padding=3, fontsize=9)
ax.bar_label(bars2, fmt="%.1f%%", padding=3, fontsize=9)
plt.tight_layout()
plt.show()

## 8. Full Per-Model Results Table

All 44 models with all four classifier accuracies on novel items, sorted by best-per-model accuracy.

In [ ]:
# Full results table
full_df = novel_df.sort_values("Best Acc", ascending=False).copy()
for c in ["RF", "LR", "SVC", "kNN", "Best Acc"]:
    full_df[c] = full_df[c].apply(lambda x: f"{x:.1%}")
full_df.index = range(1, len(full_df) + 1)
full_df[["Model", "Items", "RF", "LR", "SVC", "kNN", "Best", "Best Acc"]]

---

**Conclusion:** Random Forest is the weakest classifier on novel items across all 44 models. Linear SVC and k-NN consistently outperform it, especially on reverse-scored items where RF's axis-aligned splits break down. The atlas ships all four classifiers (`models/{model}_{rf,lr,svc,knn}_model.pkl`) so users can select the best one for their application.

For the full atlas pipeline, see the [Quick Start](https://colab.research.google.com/github/Wildertrek/survey/blob/main/notebooks/atlas_quick_start.ipynb) notebook. For embedding-space analysis (PCA, t-SNE, UMAP, SPA clustering), see the [Embedding Projector](https://colab.research.google.com/github/Wildertrek/survey/blob/main/notebooks/atlas_embedding_projector.ipynb).

> Raetano, J., Gregor, J., & Tamang, S. (2026). *A Survey and Computational Atlas of Personality Models.* ACM TIST. Under review.